# SMILES-VAE — Baseline Evaluation

Валидация обученной модели через evaluation framework.  
Запускать после `python -m experiments.train_smiles_vae`.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data.preprocessing import load_charset, get_dataloaders, ZincDataset
from models.smiles_vae import SmilesVAE
from evaluation import (
    validity, uniqueness, novelty, diversity, reconstruction_accuracy,
    compute_logp,
    plot_latent_space,
)

In [ ]:
CHECKPOINT = '../results/smiles_vae.pt'
CHARSET_PATH = '../results/charset.yaml'
DATA_PATH = '../data/zinc_250.csv'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Загрузка модели

In [ ]:
charset = load_charset(CHARSET_PATH)
n_chars = len(charset)
print(f'Charset size: {n_chars}')

model = SmilesVAE(
    max_len=120,
    n_chars=n_chars,
    hidden_dim=196,
    conv_depth=4,
    conv_start_filters=16,
    gru_depth=4,
    gru_dim=488,
).to(DEVICE)

model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval()
print('Model loaded.')

## 1. Reconstruction accuracy

Берём 1000 молекул из тест-сплита, encode -> decode, сравниваем canonical SMILES.

In [ ]:
_, val_loader, _ = get_dataloaders(
    DATA_PATH, batch_size=256, val_split=0.1, seed=42, charset=charset
)

# Используем датасет напрямую для доступа к SMILES-строкам
dataset = ZincDataset(DATA_PATH, charset=charset)
n_test = int(len(dataset) * 0.1)
test_smiles = dataset.smiles[-n_test:][:1000]

recon_smiles = model.decode_to_smiles(
    model.encode_smiles(test_smiles, charset, device=DEVICE),
    charset
)

recon_acc = reconstruction_accuracy(test_smiles, recon_smiles)
print(f'Reconstruction accuracy: {recon_acc:.3f}')

## 2. Generation metrics

Сэмплируем 1000 z ~ N(0, I) -> decode -> validity, uniqueness, novelty, diversity.

In [ ]:
N_SAMPLES = 1000

z_samples = model.sample(N_SAMPLES, device=DEVICE)
gen_smiles = model.decode_to_smiles(z_samples, charset)

val_score = validity(gen_smiles)
uniq_score = uniqueness(gen_smiles)
nov_score = novelty(gen_smiles, dataset.smiles)
div_score = diversity(gen_smiles)

print(f'Validity:    {val_score:.3f}')
print(f'Uniqueness:  {uniq_score:.3f}')
print(f'Novelty:     {nov_score:.3f}')
print(f'Diversity:   {div_score:.3f}')

baseline_metrics = {
    'reconstruction_accuracy': recon_acc,
    'validity': val_score,
    'uniqueness': uniq_score,
    'novelty': nov_score,
    'diversity': div_score,
}
print('\nBaseline metrics:', baseline_metrics)

## 3. Latent space visualization

Encode 5000 молекул, t-SNE проекция раскрашенная по logP.

In [ ]:
N_VIZ = 5000
viz_smiles = dataset.smiles[:N_VIZ]
logp_values = dataset.logp[:N_VIZ]

z_viz = model.encode_smiles(viz_smiles, charset, device=DEVICE).cpu().numpy()

fig = plot_latent_space(z_viz, logp_values, method='tsne', title='SMILES-VAE Latent Space (logP)')
plt.show()
fig.savefig('../results/latent_space_tsne.png', dpi=150, bbox_inches='tight')
print('Saved to results/latent_space_tsne.png')

## 4. Сводная таблица baseline

In [ ]:
import json, os
os.makedirs('../results', exist_ok=True)

with open('../results/baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

pd.DataFrame([baseline_metrics]).T.rename(columns={0: 'value'}).round(3)